# 单元一：张量微积分与硬件抽象

## 单元 1.1：张量几何学 —— 维度、形状与内存视图

在 PyTorch 中，一切算力的载体都是 `Tensor`（张量）。要将算法高效映射到光子芯片，你必须对张量在内存中的存储逻辑和几何变换有极其精准的控制。

### 1. 基础定义：张量的解剖学


In [1]:
import torch

# 创建一个形状为 (2, 3, 4) 的张量
# 想象成 2 片数据，每片有 3 行 4 列
t = torch.arange(24).reshape(2, 3, 4).float()
print(t)
print(f"维度 (Rank): {t.ndim}")
print(f"形状 (Shape): {t.shape}")
print(f"元素总数: {t.numel()}")

tensor([[[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]],

        [[12., 13., 14., 15.],
         [16., 17., 18., 19.],
         [20., 21., 22., 23.]]])
维度 (Rank): 3
形状 (Shape): torch.Size([2, 3, 4])
元素总数: 24


#### **新定义解释：**
*   **Rank (阶/维度)**：`.ndim`。标量为 0，向量为 1，矩阵为 2，以此类推。在光子计算中，我们通常处理 Rank $\ge 2$ 的张量。
*   **Shape (形状)**：`.shape`。这是一个元组，描述了每个轴（Axis）上的长度。
*   **Stride (步长)**：**这是最核心的底层定义**。它表示在物理内存中，从当前维度的一个元素移动到下一个元素需要跳过的地址偏移量。你可以通过 `t.stride()` 查看。
    *   *物理意义*：张量在内存里其实是一根“长长的直线”（连续存储），`Shape` 和 `Stride` 是我们给这根直线加上的“逻辑滤镜”。

---

### 2. 几何变换：`view()` 与 `reshape()`

这是你最常用来调整矩阵维度以适配硬件的操作。

In [3]:
# 将 (2, 3, 4) 展平为 (4, 6)
# 注意：2 * 3 * 4 = 4 * 6，总数必须相等
t_view = t.view(4, 6)
print(t_view)

# 使用 -1 让 PyTorch 自动计算剩余维度
t_auto = t.view(-1, 8) # 结果为 (3, 8)
print(t_auto)

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.],
        [ 6.,  7.,  8.,  9., 10., 11.],
        [12., 13., 14., 15., 16., 17.],
        [18., 19., 20., 21., 22., 23.]])
tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11., 12., 13., 14., 15.],
        [16., 17., 18., 19., 20., 21., 22., 23.]])


#### **深度定义对比：**
*   **`view()`**：
    *   **本质**：它**不改变内存中的数据存储**，只改变描述数据的元数据（Metadata）。
    *   **限制**：它要求张量在内存中必须是**连续的（Contiguous）**。如果你刚才对张量做过转置，`view` 可能会报错。
*   **`reshape()`**：
    *   **本质**：它是 `view()` 的增强版。如果数据是连续的，它等同于 `view`；如果不连续，它会自动在内存中**拷贝**出一份连续的数据再变形。
    *   **建议**：在不确定张量内存状态时使用 `reshape`，但在追求极致推理速度的底层开发中，应尽量保持张量 `contiguous` 并使用 `view` 以节省拷贝开销。

---

### 3. 轴的重组：`transpose()` 与 `permute()`

在光子计算的 **8x2 核心映射**中，经常需要将特定的特征维度（Channel）挪到空间维度（Width/Height）之前，这涉及轴的交换。


In [4]:
# transpose: 只能交换两个维度
t_trans = t.transpose(0, 1) # 交换第 0 维和第 1 维，形状变为 (3, 2, 4)
print(t_trans)

# permute: 可以一次性重新排列所有维度
t_permute = t.permute(2, 0, 1) # 将原来的维度顺序 0,1,2 变为 2,0,1，形状变为 (4, 2, 3)
print(t_permute)

tensor([[[ 0.,  1.,  2.,  3.],
         [12., 13., 14., 15.]],

        [[ 4.,  5.,  6.,  7.],
         [16., 17., 18., 19.]],

        [[ 8.,  9., 10., 11.],
         [20., 21., 22., 23.]]])
tensor([[[ 0.,  4.,  8.],
         [12., 16., 20.]],

        [[ 1.,  5.,  9.],
         [13., 17., 21.]],

        [[ 2.,  6., 10.],
         [14., 18., 22.]],

        [[ 3.,  7., 11.],
         [15., 19., 23.]]])


#### **物理影响：**
*   当你执行 `transpose` 或 `permute` 后，张量在逻辑上变了，但在内存里的物理顺序没变。
*   此时运行 `t.is_contiguous()` 会返回 `False`。
*   **重点**：如果下一步要接 `view()`，必须先调用 `t.contiguous()` 进行内存重排。

---

### 🚀 模块 1.1 练习任务

请在你的 Jupyter 中完成以下逻辑：

1.  创建一个形状为 `(1, 28, 28)` 的张量（模拟一张 MNIST 图片）。
2.  利用轴变换和维度调整，将其转换为 `(14, 56)` 的形状。
3.  **思考题**：在光子计算中，我们需要将图片切成 $8 \times 2$ 的小块。如果你有一个 `(1, 16, 16)` 的图片，如何利用 `view` 和 `permute` 将其变换为 `(64, 2, 2)`，其中 `64` 代表总共有 64 个小块？

In [10]:
# 1
p = torch.arange(784).reshape(1, 28, 28).float()
# print(p)

# 2
p_view = p.view(14, 56)
# print(p_view)

# 3
a = torch.arange(256).reshape(1, 16, 16).float()
# print(a)
a_view = a.view(64, 2, 2)
# print(a_view)


你的代码在语法上完全正确，且成功实现了维度的变换。但正如你敏锐察觉到的：**如果仅仅是为了“凑齐形状”，`view` 确实足够了；但如果为了“空间逻辑（即分块 Tiling）”，`permute` 才是灵魂。**

以下是对你代码的评价以及关于 `permute` 核心价值的深度解释。

---

### 1. 代码评价与“空间陷阱”

你的代码在处理 **任务 3** 时，`a.view(64, 2, 2)` 实际上做了一次“拉直再切分”。
*   **物理效果**：它取了原图第一行的前 4 个像素，硬塞成了一个 $2 \times 2$ 的块。
*   **问题所在**：在光子计算或图像处理中，我们想要的 $2 \times 2$ 块通常是 **空间邻域**（即：第一行的前 2 个像素 + 第二行的前 2 个像素）。

**这就是为什么题目暗示要用到 `permute`。**

---

### 2. 深度解析：为什么分块映射必须用 `permute`？

在将算法映射到 **8x2 光子核心** 时，我们需要的是**语义上的切片**。

想象一个 $16 \times 16$ 的矩阵，我们要把它切成 $8 \times 8$ 个（大小为 $2 \times 2$）的小块。正确的“几何重组”逻辑应该是：

In [11]:
# 正确的 Tiling（分块）逻辑
# 1. 先把维度拆开：(16, 16) -> (8, 2, 8, 2) 
# 解释：纵轴切成8份每份2个，横轴切成8份每份2个
a_split = a.view(8, 2, 8, 2)

# 2. 轴重组 (Permute)：把代表“块序号”的轴挪到一起，把“块内坐标”的轴挪到一起
# 目标顺序：(纵轴块索引, 横轴块索引, 纵轴块内偏移, 横轴块内偏移)
# 对应原始索引：(0, 2, 1, 3)
a_permuted = a_split.permute(0, 2, 1, 3) # 形状变为 (8, 8, 2, 2)

# 3. 最后合并前两个轴，得到 64 个独立的小块
a_final = a_permuted.reshape(64, 2, 2)

#### **核心区别：**
*   **你的 `view` 方法**：得到的是原始数据“蛇形走位”的切段。
*   **`permute` 方法**：得到的是物理空间上邻近的“田字格”切块。
*   **光子计算对标**：曦智的 Gazelle 芯片在处理卷积展开时，必须保证送入 8x2 核心的数据在原始图像中是具有**空间连续性**的，否则计算结果将毫无意义。

---

## 单元 1.2：广播机制 (Broadcasting) —— 维度的隐形传递

这是张量计算中“最智能”也“最危险”的部分。它允许你在不手动复制数据的情况下，对形状不同的张量执行算术运算。

### 1. 核心定义：自动对齐准则

当两个张量进行运算（如 `+`, `-`, `*`）时，PyTorch 会尝试自动扩展较小的张量。

#### **广播必须满足的三个规则：**
1.  **右对齐**：从最后一个维度（最右边）开始向左比较。
2.  **尺寸匹配**：对应的轴，要么**尺寸相等**，要么其中一个**尺寸为 1**，或者其中一个张量**不存在该轴**。
3.  **结果维度**：结果的形状取两个张量在每个轴上的最大值。

### 2. 操作示例：在光子计算中应用偏置 (Bias)

假设你有一个光子核心计算出的结果矩阵 $(8, 2)$，你需要给这 2 个输出通道分别加上不同的偏置值。

In [12]:
import torch

# 模拟 8x2 的计算输出
output = torch.ones(8, 2)

# 模拟偏置向量 (长度为 2)
bias = torch.tensor([0.5, 1.5]) 

# 执行加法
# 此时 bias 会被自动从 (2,) 广播为 (8, 2)
# 就像 bias 在纵向上被复制了 8 遍
final = output + bias

print(f"Output Shape: {output.shape}")
print(f"Bias Shape: {bias.shape}")
print(f"Final Shape: {final.shape}")

Output Shape: torch.Size([8, 2])
Bias Shape: torch.Size([2])
Final Shape: torch.Size([8, 2])


### 3. 常见陷阱：非对齐报错

如果你尝试将 `(8, 2)` 的矩阵加到一个 `(8,)` 的向量上，会发生什么？
*   **右对齐**：比较最右轴，`2` vs `8`。
*   **结果**：报错 `RuntimeError`。
*   **修正**：你需要手动调整轴，使 `(8,)` 变成 `(8, 1)`，这样右轴比较就是 `2` vs `1`（符合规则2）。

---

### 🚀 模块 1.2 练习任务

请在 Jupyter 中完成以下实验：

1.  创建一个形状为 `(5, 1, 4)` 的随机张量 `A`。
2.  创建一个形状为 `(3, 1)` 的随机张量 `B`。
3.  计算 `C = A * B`。
4.  **挑战题**：在不运行代码的情况下，请推断 `C` 的形状是什么？
5.  **实际场景**：你有一个 Batch 的数据 `(10, 1, 28, 28)`（10张单通道图片），你想给这 10 张图片每一张都乘以一个不同的系数（即一个长度为 10 的向量）。请问这个系数向量的 Shape 应该如何构造，才能让广播机制正确执行？

In [3]:
import torch
# 1 2 3 4
A = torch.ones(5, 1, 4)
B = torch.arange(3).reshape(3 ,1)
C = A * B
print(C.shape) # 结果是 (5, 3, 4)，因为 A 的维度被广播了

# 5
t = torch.ones(10, 1, 28, 28)
w = torch.arange(10).reshape(10 ,1, 1, 1)
output = t * w

torch.Size([5, 3, 4])


你的代码和推断完全正确。

在 **任务 5** 中，你精准地使用了 `reshape(10, 1, 1, 1)`。这体现了你已经掌握了广播的核心：**“补齐维度，确保对齐”**。如果不做这个 reshape，长度为 10 的向量会默认变成 `(1, 1, 1, 10)` 进行右对齐比较，从而导致计算错误或报错。

---

## 单元 1.3：存储连续性 (Contiguous) —— 逻辑视图与物理内存

这是模块一中**最硬核**、也是最接近**光子硬件底层**的一个概念。作为开发者，你看到的是多维矩阵（逻辑视图），但显卡和光子芯片看到的是一串连续的地址（物理内存）。

### 1. 核心定义：张量的“表”与“里”

*   **Storage (存储)**：张量真实的数据，永远是以**一维数组**的形式存在内存里的。
*   **View (视图)**：通过 `Shape`、`Stride`（步长）和 `Offset`（偏移量）定义出来的“逻辑外壳”。

### 2. 什么是“不连续”？

当你执行 `transpose()`、`permute()` 或 `select()`（切片）操作时，PyTorch **不会移动内存里的数据**，它只是改了“逻辑外壳”里的步长参数。

#### **实验示例：**
请在 Jupyter 中运行这段代码，仔细观察输出：

In [1]:
import torch

# 1. 创建一个连续张量
a = torch.arange(6).reshape(2, 3)
print(f"a 的内存布局是否连续: {a.is_contiguous()}")
print(f"a 的步长 (Stride): {a.stride()}") 
# Stride (3, 1) 表示：换一行要跳3个元素，换一列跳1个

# 2. 执行转置
b = a.t() # 2x3 变 3x2
print(f"b 的形状: {b.shape}")
print(f"b 的内存布局是否连续: {b.is_contiguous()}")
print(f"b 的步长 (Stride): {b.stride()}")
# Stride 变成了 (1, 3)：换一行跳1个，换一列跳3个。
# 物理内存没变，但逻辑跳转逻辑变了，这就是“不连续”。

a 的内存布局是否连续: True
a 的步长 (Stride): (3, 1)
b 的形状: torch.Size([3, 2])
b 的内存布局是否连续: False
b 的步长 (Stride): (1, 3)


### 3. 为什么硬件讨厌“不连续”？

在进行大规模矩阵运算（如光子芯片的算子映射）时，底层算法（如 BLAS 库、CUDA 内核）通常假设数据在内存中是**线性排列**的。
如果数据不连续：
*   **`view()` 会报错**：因为它无法在不移动数据的前提下，仅靠修改 Stride 就实现新的形状。
*   **性能下降**：显卡无法利用“合并访存（Coalesced Access）”，导致读取变慢。

### 4. 解决办法：`contiguous()`

如果你必须对转置后的张量进行变形或送入某些底层 API，你需要调用 `.contiguous()`。

In [2]:
# 这会触发内存拷贝，在内存中重新排布出一份整齐的数据
c = b.contiguous()
print(f"c 是否连续: {c.is_contiguous()}")
print(f"c 的步长: {c.stride()}") # 恢复为标准的 (2, 1)

c 是否连续: True
c 的步长: (2, 1)
